In [ ]:
%matplotlib widget
from ipywidgets import *
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import sympy as sym
from sympy import symbols, Matrix, latex, expand, I, simplify
from IPython.display import display, Math

plt.style.use('dark_background')

fontsize = 14
mpl.rcParams.update({
        "text.usetex": False,
        "figure.figsize": (9, 6),
        "figure.autolayout": True,
        "font.family": "serif",
        "font.serif": "georgia",
        # 'mathtext.fontset': 'cm',
        "lines.linewidth": 1.5,
        "font.size": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "legend.fancybox": True,
        "legend.fontsize": fontsize,
        "legend.framealpha": 0.7,
        "legend.handletextpad": 0.5,
        "legend.labelspacing": 0.2,
        "legend.loc": "best",
        "axes.edgecolor": "#b0b0b0",
        "grid.color": "#707070",  # grid color"
        "xtick.color": "#b0b0b0",
        "ytick.color": "#b0b0b0",
        "savefig.dpi": 80,
        "pdf.compression": 9,
})

In [ ]:
# mirror params
r1 = 0.9
r2 = 0.9
t1 = np.sqrt(1 - r1**2)
t2 = np.sqrt(1 - r2**2)
Ein = 1.0

# field defs
def E_trans(phi, r1, r2, t1, t2, Ein=1.0):
    return Ein * (t1 * t2 * np.exp(1j * phi)) / (1 - r1 * r2 * np.exp(2j * phi))

def E_refl(phi, r1, r2, Ein=1.0):
    return Ein * (r1 - r2 * np.exp(2j * phi)) / (1 - r1 * r2 * np.exp(2j * phi))

phi = np.linspace(-np.pi/2, 3*np.pi/2, 2000)

Et = E_trans(phi, r1, r2, t1, t2, Ein)
Er = E_refl(phi, r1, r2, Ein)

# plotting
plt.figure(figsize=(10, 5))
plt.plot(phi, np.abs(Et), label="E_trans", linewidth=2)
plt.plot(phi, np.abs(Er), label="E_refl", linewidth=2)
plt.xlabel("Phi")
plt.ylabel("Magnitude")
plt.title("1.1 Fabry Perot Sweep")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
phi0 = 0.2 * np.pi       # operating point
delta_phi = 0.05         # modulation depth
omega = 2 * np.pi * 1.0  # 1 Hz modulation

t = np.linspace(0, 3, 2000)

# original trans
E_const = E_trans(phi0, r1, r2, t1, t2, Ein)

# modulation
phi_t = phi0 + delta_phi * np.cos(omega * t)
Et_t = E_trans(phi_t, r1, r2, t1, t2, Ein)

# plotting
plt.figure(figsize=(10, 5))

plt.plot(t, np.abs(Et_t), label="Modulated", linewidth=2)
plt.plot(t, np.abs(E_const) * np.ones_like(t), '--',
         label="Non-Modulated", linewidth=2)

plt.xlabel('Time (t)')
plt.ylabel('Magnitude')
plt.title("1.2 Fabry Perot E_trans")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## modulation!

In [ ]:
# Cavity Params

c = 299792458.0
L = 1.0
Pin = 1.0
T1 = 0.3
T2 = 0.3
lam = 1064e-9
f_mod = 100.0
Omega_mod = 2 * np.pi * f_mod

# mirror params
t1 = np.sqrt(T1)
t2 = np.sqrt(T2)
r1 = np.sqrt(1 - T1)
r2 = np.sqrt(1 - T2)

Ein = np.sqrt(Pin)
k = 2 * np.pi / lam

# calcs
FSR = c / (2 * L)
Finesse = np.pi * np.sqrt(r1 * r2) / (1 - r1 * r2)
f_pole = FSR / (2 * Finesse)       
gamma = 2 * np.pi * f_pole           

print(f"FSR = {FSR:.3e} Hz")
print(f"Finesse = {Finesse:.3f}")
print(f"Cavity pole = {f_pole:.3e} Hz")

# field calculations
def E_trans(phi):
    return Ein * (t1 * t2 * np.exp(1j * phi)) / (1 - r1 * r2 * np.exp(2j * phi))

def dEtrans_dphi(phi):
    R = r1 * r2
    Et = E_trans(phi)
    return 1j * Et * (1 + R * np.exp(2j * phi)) / (1 - R * np.exp(2j * phi))

def cavity_lowpass(Omega):
    return 1 / (1 + 1j * Omega / gamma)
    
def Ptrans_over_dx(phi, Omega):
    E0 = E_trans(phi)
    dEdx = k * dEtrans_dphi(phi)
    G = np.conj(E0) * dEdx
    return G * cavity_lowpass(Omega)

# P_trans plots
phi = np.linspace(-np.pi/2, 3*np.pi/2, 2000)
G_phi = Ptrans_over_dx(phi, Omega_mod)

plt.figure(figsize=(10, 5))
plt.plot(phi, np.real(G_phi), label="Real", linewidth=2)
plt.plot(phi, np.imag(G_phi), '--', label="Im", linewidth=2)

for x in [0, np.pi]:
    plt.axvline(x, color='k', alpha=0.2)

plt.xlabel("Carrier Phi")
plt.ylabel("P_trans by del x")
plt.title("1.5 P_trans")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
phi_deg = 1.0
phi_fixed = np.deg2rad(phi_deg)

f = np.logspace(np.log10(3e3), np.log10(3e8), 2000)
Omega = 2 * np.pi * f
G_f = Ptrans_over_dx(phi_fixed, Omega)

mag = np.abs(G_f)
phase_deg = np.angle(G_f, deg=True)

plt.figure(figsize=(10, 5))
plt.loglog(f, mag, linewidth=2)
plt.ylabel("Magnitude")
plt.title("1.6 Bode Plots")
plt.grid(True, which='both')
plt.show()

plt.figure(figsize=(10, 5))
plt.semilogx(f, phase_deg, linewidth=2)
plt.xlabel("Freq (Hz)")
plt.ylabel('Phase')
plt.grid(True, which='both')
plt.show()

In [ ]:
# cavity params
T_itm = 0.10
T_etm = 0.0

r_itm = np.sqrt(1 - T_itm)
t_itm = np.sqrt(T_itm)

r_etm = np.sqrt(1 - T_etm)

# resonance cond.
Phi_c = 0.0

# DARM phase
Phi_d = np.linspace(-0.5, 0.5, 2000)

# fabry perot refl
def r_fp(Phi, r1=r_itm, t1=t_itm, r2=r_etm):
    return r1 - (t1**2 * r2 * np.exp(2j * Phi)) / (1 - r1 * r2 * np.exp(2j * Phi))


# FPMI AS response
def E_as_over_E_in(Phi_d, Phi_c=0.0):
    Phi_x = Phi_c + Phi_d
    Phi_y = Phi_c - Phi_d
    return 0.5 * (r_fp(Phi_x) - r_fp(Phi_y))

# normal michelson
def E_as_michelson(Phi_d):
    return 1j * np.sin(Phi_d)

E_fpmi = E_as_over_E_in(Phi_d, Phi_c)
E_mich = E_as_michelson(Phi_d)


plt.figure(figsize=(8, 5))
plt.plot(Phi_d, np.real(E_fpmi), label="FPMI Real")
plt.plot(Phi_d, np.real(E_mich), "--", label="Michelson Real")
plt.xlabel("Differential phase")
plt.ylabel("Real part")
plt.title("E_as/E_in")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(Phi_d, np.imag(E_fpmi), label="FPMI Imaginary")
plt.plot(Phi_d, np.imag(E_mich), "--", label="Michelson Im")
plt.xlabel("Differential phase")
plt.ylabel("Imaginary part")
plt.grid(True)
plt.legend()
plt.show()